# Chapter 3 — Adding Layers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saanikakulkarni07/learn-neural-nets/blob/main/ch03-adding-layers/exercises.ipynb)

Two big upgrades this chapter: **stacking layers** (a network becomes *deep*) and wrapping a layer in a reusable **`Layer_Dense` class** with real random initialization. We also switch from hand-typed data to the book's `nnfs` **spiral dataset** — non-linear data that a straight line can't separate.

**Goals**
1. Stack two layers by hand and respect the shape contract.
2. Generate & visualize the spiral dataset.
3. Build `Layer_Dense` from scratch and understand every init choice.
4. Run a forward pass on real data and stack two `Layer_Dense` layers.
5. Apply it to an astro-flavored feature matrix.


## 0. Setup

In [ ]:
try:
    import nnfs
except ImportError:
    !pip install nnfs -q
    import nnfs
import numpy as np
import matplotlib.pyplot as plt
from nnfs.datasets import spiral_data
nnfs.init()   # seed=0, float32 default, fixed dot -> reproducible
print('setup ok')

## 1. Concept check

1. What makes a network *deep*?
2. Layer 1 has 5 neurons. How many weights must each neuron in layer 2 have, and why?
3. In `Layer_Dense`, why are weights shaped `(n_inputs, n_neurons)` instead of `(n_neurons, n_inputs)`?
4. Why initialize weights to *small* random values (`0.01 * randn`) rather than large ones?
5. Why initialize biases to 0 — and when might that backfire (the 'dead neuron' issue)?


_Your answers:_

1. 
2. 
3. 
4. 
5. 

<details><summary>Check yourself</summary>

1. Having ≥2 hidden layers.
2. 5 weights each — one per incoming value (layer 1's outputs). Neuron count of a layer = the input size of the next layer.
3. So the forward pass is `np.dot(inputs, weights)` with no per-pass transpose.
4. Small starting weights don't dwarf the tiny updates training makes; nets like values ~[-1, 1]. Large weights slow/destabilize training.
5. 0 is a neutral start. It can backfire if a neuron's `inputs·w + b ≤ 0` for *every* sample (with a step/ReLU activation) — it always outputs 0, contributes nothing, and can't learn ('dead').
</details>

## 2. Stack two layers by hand

Reproduce the book: one batch of inputs, two sets of weights/biases. Note there's only **one** `inputs` — layer 2's input is layer 1's output. Expected layer-2 output matches below.

In [ ]:
inputs = [[1, 2, 3, 2.5],
          [2., 5., -1., 2],
          [-1.5, 2.7, 3.3, -0.8]]

weights = [[0.2, 0.8, -0.5, 1],
           [0.5, -0.91, 0.26, -0.5],
           [-0.26, -0.27, 0.17, 0.87]]
biases = [2, 3, 0.5]

weights2 = [[0.1, -0.14, 0.5],
            [-0.5, 0.12, -0.33],
            [-0.44, 0.73, -0.13]]
biases2 = [-1, 2, -0.5]

# (nnfs.init patched np.dot to require ndarrays, so wrap the raw lists)
layer1_outputs = np.dot(np.array(inputs), np.array(weights).T) + biases
layer2_outputs = np.dot(layer1_outputs, np.array(weights2).T) + biases2

print('layer1 shape:', layer1_outputs.shape, '-> layer2 input must match its 2nd dim')
print(layer2_outputs)
expected = [[ 0.5031, -1.04185, -2.03875],
            [ 0.2434, -2.7332, -5.7633 ],
            [-0.99314, 1.41254, -0.35655]]
assert np.allclose(layer2_outputs, expected, atol=1e-4)
print('✓')

## 3. The spiral dataset

Generate and visualize it. Notice the three classes interleave — **no straight line** separates them. That non-linearity is why we need neural networks.

In [ ]:
X, y = spiral_data(samples=100, classes=3)
print('X shape:', X.shape, '(samples, features)')
print('y shape:', y.shape, 'unique classes:', np.unique(y))

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='brg', s=12)
plt.title('spiral_data: 3 interleaved, non-linearly-separable classes')
plt.xlabel('feature 0'); plt.ylabel('feature 1'); plt.show()

## 4. Build `Layer_Dense`

**Exercise 4.** Implement the class: random small weights `(n_inputs, n_neurons)`, zero biases `(1, n_neurons)`, and a `forward` that does `np.dot(inputs, weights) + biases`.

In [ ]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases  = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases

# sanity: shapes
d = Layer_Dense(2, 3)
assert d.weights.shape == (2, 3)
assert d.biases.shape == (1, 3)
assert np.all(d.biases == 0)
print('weights ~ small random:\n', d.weights)
print('biases:', d.biases)
print('✓ class works')

## 5. Forward pass on real data, then stack two layers

Recreate the book's result: `dense1 = Layer_Dense(2, 3)`, forward `X`, print first 5 rows. Because `nnfs.init()` fixes the seed, your numbers should match the book (first row all zeros — the first spiral point is the origin).

In [ ]:
# Earlier cells advanced the global RNG, so reset to seed 0 to reproduce the book's
# exact numbers (this is the book's fresh-run order: init -> spiral_data -> Layer_Dense).
nnfs.init()
X, y = spiral_data(samples=100, classes=3)

dense1 = Layer_Dense(2, 3)   # 2 features -> 3 neurons
dense1.forward(X)
print('dense1.output shape:', dense1.output.shape)
print(dense1.output[:5])

book_first5 = [[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
               [-1.0475188e-04,  1.1395361e-04, -4.7983500e-05],
               [-2.7414842e-04,  3.1729150e-04, -8.6921798e-05],
               [-4.2188365e-04,  5.2666257e-04, -5.5912682e-05],
               [-5.7707680e-04,  7.1401405e-04, -8.9430439e-05]]
assert np.allclose(dense1.output[:5], book_first5, atol=1e-7)
print('✓ matches the book (seed fixed by nnfs.init)')

**Exercise 5b.** Stack a second dense layer. Its `n_inputs` must equal `dense1`'s neuron count (3). Make it output 3 values too. Confirm the chained output shape is `(300, 3)`.

In [ ]:
dense2 = Layer_Dense(3, 3)   # must take 3 inputs = dense1's 3 neurons
dense2.forward(dense1.output)
print('dense2.output shape:', dense2.output.shape)
assert dense2.output.shape == (300, 3)
print('✓ layers chained: (300,2) -> (300,3) -> (300,3)')
# Note: stacked *linear* Dense layers are still linear overall — activations (Ch.4) fix this.

## 6. 🔭 Astro bonus — a Dense layer on a feature matrix

Real astro ML uses a **design matrix** of shape `(N_objects, N_features)` — exactly the layout `Layer_Dense` expects. Here we simulate a small catalog of sources with 4 photometric features and push it through a 2-layer network (the skeleton of a classifier).

**Exercise 6.** Build `Layer_Dense(4, 8)` → `Layer_Dense(8, 3)` and forward a fake `(N, 4)` catalog. Confirm the final shape is `(N, 3)` — i.e. 3 class scores per source.

In [ ]:
rng = np.random.RandomState(42)
N = 500
# fake catalog: N sources, 4 features (e.g. u-g, g-r, r-i, i-z colors), standardized
catalog = rng.normal(size=(N, 4)).astype(np.float32)

hidden = Layer_Dense(4, 8)    # 4 features -> 8 hidden neurons
out    = Layer_Dense(8, 3)    # 8 -> 3 class scores (e.g. star/galaxy/QSO)

hidden.forward(catalog)
out.forward(hidden.output)

print('catalog      :', catalog.shape)
print('hidden.output:', hidden.output.shape)
print('out.output   :', out.output.shape, '<- 3 class scores per source')
assert out.output.shape == (N, 3)
print('✓ this is the forward-pass skeleton of a star/galaxy/QSO classifier')

### Stretch
- The outputs right now are tiny and linear (no activation), so they can't yet separate the spiral. In Ch. 4 we add ReLU/Softmax and the picture changes.
- Try `Layer_Dense` with different neuron counts and confirm the shape contract each time: layer *k*'s `n_neurons` == layer *k+1*'s `n_inputs`.
- Swap the fake catalog for a real `(N_objects, N_features)` table you have (standardize the columns first) and confirm it flows through unchanged — the layout is identical.